# Multi-Contrast DISCO-Diffusion GAN - Colab Training & Visualization

## Required File Structure
```
/content/
  mc_disco_diffusion_gan/          <-- Upload this code folder
    configs/default.yaml
    data/  models/  losses/  ...
    train.py   test.py

  fastmri_data/                    <-- Your fastMRI dataset
    knee_singlecoil_train/         (~973 files, 72.7 GB)
      file1000000.h5
      file1000001.h5 ...
    knee_singlecoil_val/           (~199 files, 14.9 GB)
      ...
    knee_singlecoil_test/          (test files, 1.4 GB)
      ...
```

**Storage note:** The full train set is 72.7 GB. Options:
- **Best:** Keep data on Google Drive, symlink to `/content/` (slower I/O but no copy needed)
- **Fast subset:** Copy only N files with `max_files` config to limit dataset size
- **Val-only test:** Use the 14.9 GB val set for quick experiments

---
## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
!pip install -q h5py scikit-image einops PyYAML matplotlib tqdm torchmetrics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

# ============================================================
# CONFIGURE THESE PATHS (edit to match your setup)
# ============================================================

# 1) Where you uploaded/copied the mc_disco_diffusion_gan code folder
PROJECT_ROOT = '/content/mc_disco_diffusion_gan'

# 2) Parent directory that CONTAINS the knee_singlecoil_train/ etc. subdirs
#    Option A: Read directly from Google Drive (no copy, slower I/O)
DATA_DIR = '/content/drive/MyDrive/fastmri_data'
#    Option B: Copy a subset to local disk first (faster I/O)
# DATA_DIR = '/content/fastmri_data'

# 3) Output directories (local disk for speed)
CHECKPOINT_DIR = '/content/checkpoints'
LOG_DIR        = '/content/logs'
OUTPUT_DIR     = '/content/outputs'

# How many .h5 files to use for training (0 = ALL, set e.g. 50 for quick test)
MAX_TRAIN_FILES = 0

# ============================================================
# Create dirs & set up Python path
# ============================================================
for d in [CHECKPOINT_DIR, LOG_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Verify the data directory has the expected subdirs
print(f"Project root:  {PROJECT_ROOT}")
print(f"Data dir:      {DATA_DIR}")
print()
for subdir in ['knee_singlecoil_train', 'knee_singlecoil_val', 'knee_singlecoil_test']:
    p = os.path.join(DATA_DIR, subdir)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.h5')])
        print(f"  {subdir}/  ->  {n} .h5 files")
    else:
        print(f"  {subdir}/  ->  NOT FOUND")

print()
!ls {PROJECT_ROOT}/

### Upload code (pick one option)

In [ ]:
# OPTION A: Copy from Google Drive
# !cp -r /content/drive/MyDrive/mc_disco_diffusion_gan /content/mc_disco_diffusion_gan

# OPTION B: Upload a zip via Colab UI
# from google.colab import files
# uploaded = files.upload()  # select mc_disco_diffusion_gan.zip
# !unzip -qo mc_disco_diffusion_gan.zip -d /content/

# OPTION C: Git clone
# !git clone <your-repo-url> /content/repo && cp -r /content/repo/mc_disco_diffusion_gan /content/

print("Uncomment ONE option above and run this cell.")

---
## 2. Visualize Training Data (.h5 files)

In [ ]:
import h5py, numpy as np, matplotlib.pyplot as plt, glob

# Pick the first .h5 from the train split
train_dir = os.path.join(DATA_DIR, 'knee_singlecoil_train')
val_dir   = os.path.join(DATA_DIR, 'knee_singlecoil_val')
test_dir  = os.path.join(DATA_DIR, 'knee_singlecoil_test')

# Use whichever split exists
for d in [train_dir, val_dir, test_dir]:
    h5_files = sorted(glob.glob(os.path.join(d, '*.h5')))
    if h5_files:
        print(f"Using: {d}/  ({len(h5_files)} files)")
        break
else:
    # Fallback: look in DATA_DIR directly
    h5_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.h5')))
    if h5_files:
        print(f"Using flat layout: {DATA_DIR}/  ({len(h5_files)} files)")
    else:
        raise FileNotFoundError(
            f"No .h5 files found in {DATA_DIR} or its subdirectories.\n"
            f"Expected: {DATA_DIR}/knee_singlecoil_train/*.h5"
        )

In [ ]:
# Inspect structure of one .h5 file
h5_path = h5_files[0]
print(f"File: {os.path.basename(h5_path)}")
print(f"Size: {os.path.getsize(h5_path) / 1e6:.1f} MB")
print("=" * 50)

with h5py.File(h5_path, 'r') as f:
    print(f"Keys: {list(f.keys())}")
    for key in f.keys():
        ds = f[key]
        if isinstance(ds, h5py.Dataset):
            print(f"  {key}: shape={ds.shape}, dtype={ds.dtype}")
    if f.attrs:
        print(f"Attrs: {dict(f.attrs)}")

In [ ]:
# Visualize slices from one volume
with h5py.File(h5_path, 'r') as f:
    # fastMRI singlecoil: 'kspace' [slices, H, W] complex64
    #                  or: 'reconstruction_rss' [slices, H, W] float32
    if 'reconstruction_rss' in f:
        img_data = f['reconstruction_rss'][:]
        src = 'reconstruction_rss'
    elif 'reconstruction_esc' in f:
        img_data = f['reconstruction_esc'][:]
        src = 'reconstruction_esc'
    elif 'kspace' in f:
        ksp = f['kspace'][:]  # [slices, H, W] complex (singlecoil)
        img_data = np.abs(np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(ksp, axes=(-2,-1)), axes=(-2,-1)), axes=(-2,-1)))
        src = 'kspace (IFFT magnitude)'
    else:
        key0 = list(f.keys())[0]
        img_data = np.abs(f[key0][:])
        src = key0

print(f"Source: {src}")
print(f"Shape: {img_data.shape}  (slices, H, W)")
print(f"Range: [{img_data.min():.4f}, {img_data.max():.4f}]")

n_slices = img_data.shape[0]
n_show = min(10, n_slices)
indices = np.linspace(0, n_slices - 1, n_show, dtype=int)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, idx in enumerate(indices):
    ax = axes.flat[i]
    sl = img_data[idx]
    vmax = np.percentile(sl, 99)
    ax.imshow(sl, cmap='gray', vmin=0, vmax=vmax)
    ax.set_title(f'Slice {idx}/{n_slices}', fontsize=9)
    ax.axis('off')
plt.suptitle(f'{os.path.basename(h5_path)} — {n_slices} slices', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# k-Space visualization + undersampling masks
mid = n_slices // 2
sl = img_data[mid]

kspace = np.fft.fftshift(np.fft.fft2(sl))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sl, cmap='gray', vmin=0, vmax=np.percentile(sl, 99))
axes[0].set_title('Image')
axes[0].axis('off')

axes[1].imshow(np.log(np.abs(kspace) + 1e-8), cmap='viridis')
axes[1].set_title('k-Space (log mag)')
axes[1].axis('off')

from data.masks import MaskFactory
mask = MaskFactory.get_mask('random', shape=sl.shape, acceleration=5, seed=42).numpy()
axes[2].imshow(np.log(np.abs(kspace * mask) + 1e-8), cmap='viridis')
axes[2].set_title('Undersampled k-Space (5x)')
axes[2].axis('off')
plt.tight_layout()
plt.show()

# All mask types
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, mt in enumerate(['random', 'radial', 'equi', 'poisson', 'gaussian']):
    m = MaskFactory.get_mask(mt, shape=sl.shape, acceleration=5, seed=42)
    axes[i].imshow(m.numpy(), cmap='gray')
    frac = m.sum().item() / m.numel() * 100
    axes[i].set_title(f'{mt} ({frac:.1f}%)', fontsize=10)
    axes[i].axis('off')
plt.suptitle('Mask Types (5x acceleration)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Test Dataset Pipeline
Verify the full data loading pipeline works before training.

In [ ]:
import torch
from utils.config import load_config
from data.datasets import MultiContrastMRIDataset
from data.transforms import complex_magnitude
import numpy as np, matplotlib.pyplot as plt

# Build config with Colab path overrides
overrides = [
    f'paths.data_root={DATA_DIR}',
    f'paths.checkpoint_dir={CHECKPOINT_DIR}',
    f'paths.log_dir={LOG_DIR}',
    f'paths.output_dir={OUTPUT_DIR}',
]
if MAX_TRAIN_FILES > 0:
    overrides.append(f'data.max_files={MAX_TRAIN_FILES}')

config = load_config('configs/default.yaml', overrides=overrides)

print(f"Dataset:      {config.data.dataset}")
print(f"Image size:   {config.data.image_size}")
print(f"Acceleration: {config.inference.acceleration}x")
print(f"Mask type:    {config.inference.mask_type}")
print(f"Max files:    {config.data.max_files or 'ALL'}")
print()

# Verify all 3 splits load correctly
for split_name in ['train', 'val', 'test']:
    try:
        ds = MultiContrastMRIDataset(config, split=split_name)
        print(f"{split_name:5s} -> {len(ds):6d} slices")
    except FileNotFoundError:
        print(f"{split_name:5s} -> NOT FOUND (knee_singlecoil_{split_name}/ missing)")
    except Exception as e:
        print(f"{split_name:5s} -> ERROR: {e}")

# Keep references for visualization
train_ds = MultiContrastMRIDataset(config, split='train')
test_ds  = MultiContrastMRIDataset(config, split='test')

In [ ]:
# Visualize a TRAIN sample and a TEST sample side by side
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, (ds, label) in enumerate([(train_ds, 'TRAIN'), (test_ds, 'TEST')]):
    sample = ds[len(ds)//2]
    x0   = sample['x0'].unsqueeze(0)
    x_zf = sample['x_zf'].unsqueeze(0)
    mask = sample['mask']

    gt = complex_magnitude(x0)[0, 0].numpy()     # contrast 1 magnitude
    zf = complex_magnitude(x_zf)[0, 0].numpy()
    err = np.abs(gt - zf)
    vmax = np.percentile(gt, 99)

    axes[row,0].imshow(gt, cmap='gray', vmin=0, vmax=vmax)
    axes[row,0].set_title('Ground Truth' if row==0 else '')
    axes[row,0].set_ylabel(f'{label}\nslice {len(ds)//2}', fontsize=11)

    axes[row,1].imshow(mask[0].numpy(), cmap='gray')
    axes[row,1].set_title(f'Mask ({config.inference.acceleration}x)' if row==0 else '')

    axes[row,2].imshow(zf, cmap='gray', vmin=0, vmax=vmax)
    axes[row,2].set_title('Zero-Fill Recon' if row==0 else '')

    axes[row,3].imshow(err*5, cmap='hot', vmin=0, vmax=vmax)
    axes[row,3].set_title('Error (5x)' if row==0 else '')

    for a in axes[row]: a.axis('off')

plt.suptitle('Train vs Test: GT | Mask | Zero-Fill | Error', fontsize=14)
plt.tight_layout()
plt.show()
print("Dataset pipeline OK for both train and test!")

---
## 4. Verify Model Fits in GPU

In [ ]:
from models.generator import MCDISCOGenerator
from models.discriminator import TimeConditionedDiscriminator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

gen = MCDISCOGenerator(config).to(device)
disc = TimeConditionedDiscriminator(
    in_channels=config.model.in_channels,
    base_channels=config.model.base_channels,
    num_basis=config.model.disco.num_basis,
    time_emb_dim=config.model.time_emb_dim,
).to(device)

print(f"Generator params:     {sum(p.numel() for p in gen.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in disc.parameters()):,}")

# Forward pass test
B, H, W = config.training.batch_size, config.data.image_size, config.data.image_size
x = torch.randn(B, 4, H, W, device=device)
z = torch.randn(B, config.model.latent_dim, device=device)
t = torch.randint(1, config.diffusion.T+1, (B,), device=device)

with torch.no_grad():
    out = gen(x, z, t)
    d_out = disc(out, x, t)
print(f"Gen out:  {out.shape}")
print(f"Disc out: {d_out.shape}")

del gen, disc, x, z, t, out, d_out
torch.cuda.empty_cache()

if torch.cuda.is_available():
    print(f"GPU Memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} / {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")
print("\nModel test PASSED!")

---
## 5. Train

In [ ]:
# Build the override string
max_files_override = f'data.max_files={MAX_TRAIN_FILES}' if MAX_TRAIN_FILES > 0 else ''

!cd {PROJECT_ROOT} && python train.py \
    --config configs/default.yaml \
    --auto_resume \
    --overrides \
        paths.data_root={DATA_DIR} \
        paths.checkpoint_dir={CHECKPOINT_DIR} \
        paths.log_dir={LOG_DIR} \
        paths.output_dir={OUTPUT_DIR} \
        training.num_epochs=200 \
        model.base_channels=32 \
        data.image_size=320 \
        {max_files_override}

In [ ]:
# === Checkpoint management ===

# List checkpoints
!ls -lhtr {CHECKPOINT_DIR}/

# Save to Drive (survives Colab restarts)
# !mkdir -p /content/drive/MyDrive/mc_disco_gan/checkpoints
# !cp -v {CHECKPOINT_DIR}/*.pth /content/drive/MyDrive/mc_disco_gan/checkpoints/

# Restore from Drive after restart
# !cp -v /content/drive/MyDrive/mc_disco_gan/checkpoints/*.pth {CHECKPOINT_DIR}/

---
## 6. Visualize Training Curves

In [ ]:
curves_path = os.path.join(LOG_DIR, 'training_curves.png')
if os.path.exists(curves_path):
    from IPython.display import Image, display
    display(Image(filename=curves_path))
else:
    print("Training curves not found yet (saved after training completes).")
    !ls -la {LOG_DIR}/ 2>/dev/null || echo "Log dir empty"

---
## 7. Run Test / Inference

In [ ]:
import glob

# Find best or latest checkpoint
best_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
if not os.path.exists(best_ckpt):
    ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch_*.pth')))
    best_ckpt = ckpts[-1] if ckpts else None

if best_ckpt:
    print(f"Checkpoint: {best_ckpt}")
else:
    print("No checkpoint found! Train first.")

In [ ]:
# Run test on the knee_singlecoil_test split
if best_ckpt:
    !cd {PROJECT_ROOT} && python test.py \
        --config configs/default.yaml \
        --checkpoint {best_ckpt} \
        --split test \
        --max_figures 10 \
        --overrides \
            paths.data_root={DATA_DIR} \
            paths.checkpoint_dir={CHECKPOINT_DIR} \
            paths.log_dir={LOG_DIR} \
            paths.output_dir={OUTPUT_DIR}

---
## 8. Visualize Test Results

In [ ]:
# Show saved comparison figures
from IPython.display import Image, display
import glob

test_figs = sorted(glob.glob(os.path.join(OUTPUT_DIR, '**', 'comparison_*.png'), recursive=True))
print(f"Found {len(test_figs)} comparison figures")

for fig_path in test_figs[:5]:
    print(f"\n{os.path.basename(fig_path)}")
    display(Image(filename=fig_path, width=900))

In [ ]:
# Print metrics summary
for sf in sorted(glob.glob(os.path.join(OUTPUT_DIR, '**', 'metrics_summary.txt'), recursive=True)):
    print(f"\n{'='*60}")
    with open(sf) as f:
        print(f.read())

In [ ]:
# Interactive: load checkpoint and run inference on a TEST sample
import torch
import numpy as np, matplotlib.pyplot as plt
from utils.config import load_config
from utils.checkpoint import load_checkpoint
from models.generator import MCDISCOGenerator
from models.noise_schedule import DiffusionNoiseSchedule
from inference.data_consistency import DataConsistencyLayer, NoiseMixedDataConsistencyLayer
from inference.pipeline import ThreeStageInference
from data.datasets import MultiContrastMRIDataset
from data.transforms import complex_magnitude
from metrics.reconstruction import compute_all_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = load_config('configs/default.yaml', overrides=[
    f'paths.data_root={DATA_DIR}',
    f'paths.checkpoint_dir={CHECKPOINT_DIR}',
])

if best_ckpt and os.path.exists(best_ckpt):
    gen = MCDISCOGenerator(config).to(device)
    ref_gen = MCDISCOGenerator(config).to(device)
    load_checkpoint(best_ckpt, gen, ref_gen, device=device)
    gen.eval(); ref_gen.eval()

    schedule = DiffusionNoiseSchedule(T=config.diffusion.T,
        beta_start=config.diffusion.beta_start, beta_end=config.diffusion.beta_end).to(device)
    refine_schedule = DiffusionNoiseSchedule(T=config.diffusion.T_refine,
        beta_start=config.diffusion.beta_start, beta_end=config.diffusion.beta_end/4).to(device)

    pipeline = ThreeStageInference(
        gen, ref_gen, schedule, refine_schedule,
        DataConsistencyLayer().to(device), NoiseMixedDataConsistencyLayer().to(device),
        config, device=device)

    # Load from knee_singlecoil_test
    test_ds = MultiContrastMRIDataset(config, split='test')
    sample = test_ds[0]
    x0   = sample['x0'].unsqueeze(0).to(device)
    y_obs = sample['y_obs'].unsqueeze(0).to(device)
    mask  = sample['mask'].unsqueeze(0).to(device)
    x_zf  = sample['x_zf'].unsqueeze(0).to(device)

    print("Running 3-stage inference on test sample...")
    with torch.no_grad():
        results = pipeline.run(y_obs, mask, run_dgp=True, run_refinement=True)

    # --- Plot all stages ---
    stages = {'GT': x0, 'Zero-Fill': x_zf, 'Stage1': results['stage1'],
              'Stage2 (DGP)': results['stage2'], 'Stage3 (Final)': results['stage3']}

    fig, axes = plt.subplots(2, len(stages), figsize=(4*len(stages), 8))
    for c in range(2):
        gt_mag = complex_magnitude(x0)[0,c].cpu().numpy()
        vmax = np.percentile(gt_mag, 99)
        for col, (name, t) in enumerate(stages.items()):
            mag = complex_magnitude(t)[0,c].cpu().numpy()
            axes[c,col].imshow(mag, cmap='gray', vmin=0, vmax=vmax)
            if c==0: axes[c,col].set_title(name, fontsize=10)
            if col==0: axes[c,col].set_ylabel(f'Contrast {c+1}', fontsize=11)
            axes[c,col].axis('off')
    plt.suptitle('3-Stage Inference on TEST Sample', fontsize=14)
    plt.tight_layout(); plt.show()

    # --- Error maps ---
    fig, axes = plt.subplots(2, len(stages)-1, figsize=(4*(len(stages)-1), 8))
    for c in range(2):
        gt_mag = complex_magnitude(x0)[0,c].cpu().numpy()
        vmax = np.percentile(gt_mag, 99) * 0.2
        for col, (name, t) in enumerate(list(stages.items())[1:]):
            err = np.abs(gt_mag - complex_magnitude(t)[0,c].cpu().numpy())
            axes[c,col].imshow(err, cmap='hot', vmin=0, vmax=vmax)
            if c==0: axes[c,col].set_title(f'{name} Error', fontsize=10)
            axes[c,col].axis('off')
    plt.suptitle('Error Maps (TEST sample)', fontsize=14)
    plt.tight_layout(); plt.show()

    # --- Metrics ---
    print(f"\n{'Method':20s}  {'PSNR':>8s}  {'SSIM':>8s}  {'NMSE':>8s}")
    print("-" * 50)
    for name, t in stages.items():
        if name == 'GT': continue
        m = compute_all_metrics(t, x0)
        print(f"{name:20s}  {m['psnr']:8.2f}  {m['ssim']:8.4f}  {m['nmse']:8.4f}")

    del gen, ref_gen, pipeline; torch.cuda.empty_cache()
else:
    print("No checkpoint. Train first!")

---
## 9. Save Everything to Google Drive

In [ ]:
DRIVE_SAVE = '/content/drive/MyDrive/mc_disco_gan'
!mkdir -p {DRIVE_SAVE}
!cp -r {CHECKPOINT_DIR} {DRIVE_SAVE}/
!cp -r {LOG_DIR} {DRIVE_SAVE}/
!cp -r {OUTPUT_DIR} {DRIVE_SAVE}/
print(f"\nSaved to {DRIVE_SAVE}/")
!du -sh {DRIVE_SAVE}/*